# Aula 13 — Estatísticas, categorias e agrupamentos

**Módulo 4 — Pandas e Análise de Dados**

## Objetivos da aula

- Resumir dados numéricos com `describe()`.
- Analisar variáveis categóricas com `value_counts()`.
- Comparar grupos de observações com `groupby()`.

---

## 1. Recriando a base de dados

In [1]:
import numpy as np
import pandas as pd

np.random.seed(42)

n = 500
tipos_equipamento = ["Motor", "Bomba", "Compressor", "Ventilador"]

dados = pd.DataFrame({
    "equipamento_id": [f"EQ-{i:04d}" for i in range(1, n + 1)],
    "tipo_equipamento": np.random.choice(tipos_equipamento, size=n),
    "temperatura": np.round(np.random.normal(70, 12, size=n), 1),
    "pressao": np.round(np.random.normal(5.5, 1.3, size=n), 2),
    "vibracao": np.round(np.random.normal(2.4, 1.1, size=n), 2),
    "horas_operacao": np.random.randint(0, 10000, size=n),
})


def definir_status(linha):
    critico = (linha["temperatura"] >= 90) or (linha["vibracao"] >= 4.5) or (linha["pressao"] >= 8) or (linha["pressao"] <= 2)
    alerta = (linha["temperatura"] >= 80) or (linha["vibracao"] >= 3.5) or (linha["pressao"] >= 7) or (linha["pressao"] <= 3)
    if critico:
        return "critico"
    elif alerta:
        return "alerta"
    else:
        return "normal"


dados["status"] = dados.apply(definir_status, axis=1)
dados.to_csv("sensores_industriais.csv", index=False)

df = pd.read_csv("sensores_industriais.csv")
df.head()


,equipamento_id,tipo_equipamento,temperatura,pressao,vibracao,horas_operacao,status
0,EQ-0001,Compressor,59.8,6.42,3.01,1958,normal
1,EQ-0002,Ventilador,51.8,6.08,1.33,6344,normal
2,EQ-0003,Motor,64.6,5.03,2.52,5779,normal
3,EQ-0004,Compressor,80.3,7.01,0.93,6144,alerta
4,EQ-0005,Compressor,72.6,4.09,1.74,5063,normal


## 2. `describe()`: resumo estatístico rápido

O método `describe()` calcula, de uma só vez, as principais estatísticas (contagem, média, desvio padrão, mínimo, máximo e quartis) para todas as colunas numéricas — o equivalente a aplicar tudo o que vimos na Aula 10, mas em uma tabela pronta.

In [2]:
df.describe()


,temperatura,pressao,vibracao,horas_operacao
count,500.000000,500.000000,500.000000,500.000000
mean,69.920200,5.641180,2.471200,4767.772000
std,12.099887,1.283501,1.098712,2833.143821
min,31.100000,1.730000,-0.810000,6.000000
25%,61.475000,4.780000,1.720000,2219.250000
50%,69.900000,5.695000,2.440000,4698.000000
75%,77.825000,6.455000,3.172500,7152.000000
max,116.200000,8.920000,5.910000,9984.000000


**Como ler os quartis:** `25%`, `50%` e `75%` são os valores abaixo dos quais estão 25%, 50% (a mediana) e 75% dos dados, respectivamente. Eles dão uma ideia da distribuição sem precisar olhar cada valor individualmente.

Cada linha significa:

* **`count`** → quantidade de valores válidos, isto é, valores que **não são `NaN`**.
* **`mean`** → média aritmética dos valores.
* **`std`** → desvio padrão, indicando quanto os valores estão dispersos em torno da média.
* **`min`** → menor valor encontrado.
* **`25%`** → primeiro quartil, (Q_1). Aproximadamente 25% dos valores estão abaixo ou iguais a ele.
* **`50%`** → segundo quartil, (Q_2), que é a **mediana**.
* **`75%`** → terceiro quartil, (Q_3). Aproximadamente 75% dos valores estão abaixo ou iguais a ele.
* **`max`** → maior valor encontrado.

Uma forma visual de entender os quartis:

```text
mínimo        Q1         mediana        Q3         máximo
  |-----------|-------------|------------|-------------|
 min         25%           50%          75%           max
```

Ou seja, `describe()` responde rapidamente perguntas como:

> Quantos dados existem?
> Qual é o valor típico?
> Quanto os dados variam?
> Qual é o menor e o maior valor?
> Como os valores estão distribuídos?

In [3]:
# describe() também funciona em uma única coluna
df["temperatura"].describe()


count    500.000000
mean      69.920200
std       12.099887
min       31.100000
25%       61.475000
50%       69.900000
75%       77.825000
max      116.200000
Name: temperatura, dtype: float64

## 3. `value_counts()`: contando categorias

Para colunas **categóricas** (texto), `value_counts()` conta quantas vezes cada valor aparece — essencial para entender a composição da base.

In [4]:
print("Quantidade de leituras por tipo de equipamento:")
print(df["tipo_equipamento"].value_counts())

print("\nQuantidade de leituras por status:")
print(df["status"].value_counts())


Quantidade de leituras por tipo de equipamento:
tipo_equipamento
Ventilador    148
Compressor    122
Motor         122
Bomba         108
Name: count, dtype: int64

Quantidade de leituras por status:
status
normal     280
alerta     161
critico     59
Name: count, dtype: int64


In [5]:
# normalize=True mostra proporções (%) em vez de contagens absolutas
df["status"].value_counts(normalize=True).round(3)


status
normal     0.560
alerta     0.322
critico    0.118
Name: proportion, dtype: float64

## 4. `groupby()`: comparando grupos

`groupby()` divide o `DataFrame` em grupos com base em uma coluna categórica, permitindo calcular estatísticas **separadamente para cada grupo** — por exemplo, comparar a temperatura média entre os diferentes tipos de equipamento.

In [6]:
media_por_tipo = df.groupby("tipo_equipamento")["temperatura"].mean()
print(media_por_tipo.round(1))


tipo_equipamento
Bomba         69.1
Compressor    68.5
Motor         71.5
Ventilador    70.4
Name: temperatura, dtype: float64


### Agrupando com múltiplas estatísticas ao mesmo tempo: `.agg()`

In [7]:
resumo_por_tipo = df.groupby("tipo_equipamento")["temperatura"].agg(["mean", "std", "min", "max", "count"])
resumo_por_tipo.round(1)


,mean,std,min,max,count
tipo_equipamento,,,,,
Bomba,69.1,12.3,40.3,95.8,108
Compressor,68.5,11.7,37.6,96.3,122
Motor,71.5,12.5,31.1,116.2,122
Ventilador,70.4,11.8,38.2,100.9,148


### Agrupando por mais de uma coluna

Podemos passar uma lista de colunas para `groupby()`, criando grupos combinando cada categoria com cada outra — por exemplo, "tipo de equipamento" cruzado com "status".

In [8]:
contagem_tipo_status = df.groupby(["tipo_equipamento", "status"])["equipamento_id"].count()
print(contagem_tipo_status)


tipo_equipamento  status 
Bomba             alerta     33
                  critico    12
                  normal     63
Compressor        alerta     38
                  critico    12
                  normal     72
Motor             alerta     40
                  critico    15
                  normal     67
Ventilador        alerta     50
                  critico    20
                  normal     78
Name: equipamento_id, dtype: int64


### `pd.crosstab`: uma forma mais visual de cruzar duas categorias

`crosstab` monta uma tabela cruzando duas colunas categóricas, com uma linha por categoria da primeira e uma coluna por categoria da segunda — mais fácil de ler do que o `groupby` acima.

In [9]:
import pandas as pd

pd.crosstab(df["tipo_equipamento"], df["status"])


status,alerta,critico,normal
tipo_equipamento,,,
Bomba,33,12,63
Compressor,38,12,72
Motor,40,15,67
Ventilador,50,20,78


## 5. Ordenando resultados agregados

O resultado de um `groupby` também pode ser ordenado com `sort_values()`, para destacar rapidamente os grupos com maiores (ou menores) valores.

In [10]:
media_por_tipo.sort_values(ascending=False)


tipo_equipamento
Motor         71.459836
Ventilador    70.427027
Bomba         69.090741
Compressor    68.500000
Name: temperatura, dtype: float64

## 6. Resumo da aula

- `describe()` resume estatisticamente as colunas numéricas de uma vez.
- `value_counts()` conta a frequência de cada categoria em uma coluna; `normalize=True` mostra proporções.
- `groupby("coluna")` divide os dados em grupos; combinado com `.mean()`, `.agg([...])` etc., compara grupos entre si.
- `pd.crosstab()` cruza duas colunas categóricas em forma de tabela.
- `sort_values()` ordena os resultados agregados, útil para identificar rapidamente os extremos.

### Exercício sugerido

Use `groupby("status")` para calcular a média de `vibracao` e `horas_operacao` para cada status (`normal`, `alerta`, `crítico`), usando `.agg(["mean", "count"])`. O que essa comparação sugere sobre a relação entre tempo de uso e status do equipamento?
